# 06 — Map playground
Строим план на синтетической Москве и рисуем настоящую дорожную геометрию через OSRM. Один запрос выполняется на маршрут, ответы сохраняются в `solver/.cache/route_geometry.json`. Для работы дорожного режима нужен интернет; без него уберите `geometry_provider` и получите прямые линии.

In [ ]:
import html as html_lib
from pathlib import Path
from IPython.display import HTML, display
from routing_opt.route_geometry import CachedRouteGeometry, OsrmRouteGeometryClient
from routing_opt.solvers import OrToolsSolver
from routing_opt.synthetic import SyntheticConfig, generate_synthetic_problem
from routing_opt.validation import validate_solution
from routing_opt.visualization import save_gantt, save_route_map


In [ ]:
problem = generate_synthetic_problem(SyntheticConfig(job_count=20, seed=42))
solution = OrToolsSolver().solve(problem, time_limit_sec=1, seed=42)
report = validate_solution(problem, solution)
report.valid, solution.metrics

In [ ]:
root = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
output = root / 'solver' / 'outputs' / 'map-playground'
geometry = CachedRouteGeometry(
    OsrmRouteGeometryClient('https://router.project-osrm.org'),
    root / 'solver' / '.cache' / 'route_geometry.json',
)
map_path = output / 'road-map.html'
save_route_map(problem, solution, map_path, geometry_provider=geometry)
save_gantt(solution, output / 'gantt.html')
map_document = map_path.read_text(encoding='utf-8')
embedded = html_lib.escape(map_document, quote=True)
display(HTML(f"<iframe srcdoc='{embedded}' style='width:100%;height:700px;border:0' loading='lazy'></iframe>"))

## Поиграться
Меняйте `job_count`, `seed`, `time_limit_sec`, solver и параметры генератора. Чтобы сравнить прямые и дорожные линии, второй раз вызовите `save_route_map(problem, solution, ..., geometry_provider=None)`.